In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For model building
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet # Added ElasticNet
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv('AmesHousing.csv')

In [ ]:
# Load the dataset
df = pd.read_csv('AmesHousing.csv')

# Drop irrelevant columns immediately
df = df.drop(['Order', 'PID', 'Utilities'], axis=1)

# Separate features (X_temp) from the target (SalePrice) for column identification
X_temp = df.drop('SalePrice', axis=1)

# Identify numerical columns based on dtype
numerical_cols = X_temp.select_dtypes(include=np.number).columns.tolist()

# Identify all object (string) columns initially
object_cols = X_temp.select_dtypes(include='object').columns.tolist()

# Define specific ordinal columns and their intended orders based on domain knowledge
# 'None' is explicitly included in orders where it represents absence/lowest rank
ordinal_features_and_orders_manual = [
    ('Street', ['Grvl', 'Pave']),
    ('Alley', ['None', 'Grvl', 'Pave']),
    ('Lot Shape', ['IR3', 'IR2', 'IR1', 'Reg']),
    ('Land Contour', ['Low', 'Bnk', 'HLS', 'Lvl']),
    ('Land Slope', ['Sev', 'Mod', 'Gtl']),
    ('Exter Qual', ['Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Exter Cond', ['Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Bsmt Qual', ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Bsmt Cond', ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Bsmt Exposure', ['None', 'No', 'Mn', 'Av', 'Gd']),
    ('BsmtFin Type 1', ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']),
    ('BsmtFin Type 2', ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']),
    ('Heating QC', ['Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Kitchen Qual', ['Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Functional', ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']),
    ('Fireplace Qu', ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Garage Finish', ['None', 'Unf', 'RFn', 'Fin']),
    ('Garage Qual', ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Garage Cond', ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Paved Drive', ['N', 'P', 'Y']),
    ('Pool QC', ['None', 'Fa', 'TA', 'Gd', 'Ex']),
    ('Fence', ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'])
]
# Extract column names for ordinal encoder
ordinal_cols = [col for col, _ in ordinal_features_and_orders_manual]
# Extract category orders for ordinal encoder
ordinal_categories = [order for _, order in ordinal_features_and_orders_manual]

# Nominal columns are all object columns not identified as ordinal
nominal_cols = [col for col in object_cols if col not in ordinal_cols]


print("Identified Numerical Columns:", numerical_cols)
print("\nIdentified Ordinal Columns:", ordinal_cols)
print("\nIdentified Nominal Categorical Columns:", nominal_cols)

#### Analysis: Data Loading and Initial Column Identification
- Data Ingestion: Loaded the AmesHousing.csv dataset.
- Initial Cleanup: Dropped immediately irrelevant columns (Order, PID, Utilities) to streamline the dataset.
- Feature-Target Separation: Divided the dataset into features (X_temp) and the target variable (SalePrice).
- Column Categorization: Automatically identified numerical columns.
- Manually defined and extracted ordinal categorical columns with their specific ordered categories based on domain knowledge. Remaining object-type columns were classified as nominal categorical.
- Importance: This step is crucial for applying appropriate preprocessing techniques (e.g., scaling for numerical, specific encoding for ordinal/nominal) later.

In [ ]:
# Missing Value Imputation: Categorical/Ordinal features (fill 'None')
# Applies to features where 'NA' means 'no such feature'
for col in ['Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Fireplace Qu', 'Garage Type',
            'Garage Finish', 'Garage Qual', 'Garage Cond', 'Bsmt Qual', 'Bsmt Cond',
            'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Mas Vnr Type']:
    if col in df.columns: # Check if column exists (e.g., 'Pool QC' might not always be present)
        df[col] = df[col].fillna('None')

# Missing Value Imputation: Numerical features (fill 0)
# Applies to numerical features where 'NA' means 'zero value' or 'absence'
for col in ['Garage Yr Blt', 'Garage Area', 'Garage Cars', 'BsmtFin SF 1', 'BsmtFin SF 2',
            'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Mas Vnr Area',
            'Pool Area', 'Misc Val']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Lot Frontage: Impute with median of the neighborhood for spatial consistency
if 'Lot Frontage' in df.columns and 'Neighborhood' in df.columns:
    df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(lambda x: x.fillna(x.median()))

# Fallback for Lot Frontage if any NaNs remain after neighborhood imputation
if 'Lot Frontage' in df.columns and df['Lot Frontage'].isnull().any():
    global_median_lot_frontage = df['Lot Frontage'].median()
    df['Lot Frontage'] = df['Lot Frontage'].fillna(global_median_lot_frontage)


# Remaining isolated missing values (Electrical, MS Zoning) - impute with mode
# These are typically nominal categorical features with few missing values
for col in ['Electrical', 'MS Zoning']:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# Special handling for Garage Yr Blt: if 0 (missing or no garage) but garage area exists, use Year Built
if 'Garage Yr Blt' in df.columns and 'Garage Area' in df.columns and 'Year Built' in df.columns:
    df.loc[(df['Garage Yr Blt'] == 0) & (df['Garage Area'] > 0), 'Garage Yr Blt'] = df['Year Built']

# Outlier Removal: Remove houses with excessively large living areas to improve model robustness
if 'Gr Liv Area' in df.columns:
    df = df[df['Gr Liv Area'] < 4500]

print("Data cleaning complete. Missing values remaining (if any, should be minimal for key columns):")
print(df.isnull().sum()[df.isnull().sum() > 0])

#### Analysis: Cleaning
- Missing Value Strategy (Categorical/Ordinal): NaNs in columns like Pool QC, Alley, Bsmt Qual, etc., were imputed with the string 'None', signifying the absence of that feature.
- Missing Value Strategy (Numerical): NaNs in numerical columns such as Garage Area, Total Bsmt SF, etc., were imputed with 0, indicating the absence of a value or a zero quantity.
- Contextual Imputation (Lot Frontage): NaNs in Lot Frontage were filled using the median Lot Frontage of their respective neighborhoods to preserve spatial relationships. A global median fallback was included.
- Mode Imputation: Isolated missing values in columns like Electrical and MS Zoning were filled with their respective modes.
- Data Consistency: Corrected Garage Yr Blt values where 0 was present but a garage existed, using Year Built for consistency.
- Outlier Removal: Extreme outliers in Gr Liv Area (gross living area) were removed to prevent them from skewing model training and improving robustness.

In [ ]:
# Create new features by combining existing square footage
if all(col in df.columns for col in ['1st Flr SF', '2nd Flr SF', 'Total Bsmt SF']):
    df['TotalSF'] = df['1st Flr SF'] + df['2nd Flr SF'] + df['Total Bsmt SF']

# Create total bathroom count
if all(col in df.columns for col in ['Full Bath', 'Half Bath', 'Bsmt Full Bath', 'Bsmt Half Bath']):
    df['TotalBath'] = df['Full Bath'] + (0.5 * df['Half Bath']) + df['Bsmt Full Bath'] + (0.5 * df['Bsmt Half Bath'])

# Create total porch square footage
if all(col in df.columns for col in ['Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch']):
    df['TotalPorchSF'] = df['Wood Deck SF'] + df['Open Porch SF'] + df['Enclosed Porch'] + df['3Ssn Porch'] + df['Screen Porch']

# Create age-related features
if all(col in df.columns for col in ['Yr Sold', 'Year Built']):
    df['HouseAge'] = df['Yr Sold'] - df['Year Built']

if all(col in df.columns for col in ['Yr Sold', 'Year Remod/Add']):
    df['RemodelAge'] = df['Yr Sold'] - df['Year Remod/Add']
    # Ensure remodel age is not negative (if remodel year is after sold year)
    df.loc[df['RemodelAge'] < 0, 'RemodelAge'] = 0

if all(col in df.columns for col in ['Yr Sold', 'Garage Yr Blt']):
    df['GarageAge'] = df['Yr Sold'] - df['Garage Yr Blt']
    # Ensure garage age is not negative
    df.loc[df['GarageAge'] < 0, 'GarageAge'] = 0

# Create interaction features
if all(col in df.columns for col in ['Overall Qual', 'Gr Liv Area']):
    df['OverallQual_GrLivArea'] = df['Overall Qual'] * df['Gr Liv Area']
if all(col in df.columns for col in ['Overall Qual', 'TotalSF']):
    df['OverallQual_TotalSF'] = df['Overall Qual'] * df['TotalSF']

# Create binary existence features (0 or 1)
if 'Pool Area' in df.columns:
    df['HasPool'] = (df['Pool Area'] > 0).astype(int)
if '2nd Flr SF' in df.columns:
    df['Has2ndFloor'] = (df['2nd Flr SF'] > 0).astype(int)
if 'Garage Area' in df.columns:
    df['HasGarage'] = (df['Garage Area'] > 0).astype(int)
if 'Total Bsmt SF' in df.columns:
    df['HasBsmt'] = (df['Total Bsmt SF'] > 0).astype(int)
if 'Fireplaces' in df.columns:
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)

# Create ratio feature (average square footage per room)
if all(col in df.columns for col in ['Gr Liv Area', 'TotRms AbvGrd']):
    df['AvgRoomSF'] = df.apply(lambda row: row['Gr Liv Area'] / (row['TotRms AbvGrd'] if row['TotRms AbvGrd'] > 0 else 1), axis=1)

# Final type conversion for all engineered features to ensure they are numeric
# and handle any NaNs that might arise from coercion (e.g., division by zero)
engineered_features_list = [
    'TotalSF', 'TotalBath', 'TotalPorchSF', 'HouseAge', 'RemodelAge', 'GarageAge',
    'OverallQual_GrLivArea', 'OverallQual_TotalSF', 'HasPool', 'Has2ndFloor',
    'HasGarage', 'HasBsmt', 'HasFireplace', 'AvgRoomSF'
]
for col in engineered_features_list:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("Feature engineering complete. First 5 rows with new features:")
print(df.head())

### Analysis: Feature Engineering
- Combined Area Features: Created TotalSF (total square footage), TotalBath (total bathrooms), and TotalPorchSF (total porch area) to provide comprehensive size metrics.
- Age-Related Features: Calculated HouseAge, RemodelAge, and GarageAge from year columns, providing insight into the property's age and renovation status. Negative ages were corrected to zero.
- Interaction Features: Introduced OverallQual_GrLivArea and OverallQual_TotalSF by multiplying overall quality with area, capturing synergistic effects where high quality combined with large size might be particularly valuable.
- Binary Existence Features: Created HasPool, Has2ndFloor, HasGarage, HasBsmt, and HasFireplace (0 or 1 flags) to indicate the presence or absence of key amenities.
- Ratio Feature: Derived AvgRoomSF (average square footage per room) to reflect spaciousness.
- Type Coercion: Ensured all engineered features were numeric, filling any NaNs resulting from calculations with 0

In [ ]:
# Separate features (X) and target (y)
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Preprocessing Pipeline Setup ---
# Define the ColumnTransformer using the identified column lists from Cell 2
# Note: For simple linear regression, we'll manually select a feature from X_train
# before passing it to the pipeline that only has a scaler.

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')), # Impute NaNs with the median
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('ord', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols)
    ],
    remainder='passthrough'
)

# --- Model Pipelines and Training ---

print("\n--- Starting Model Training ---")

# 1. Simple Linear Regression Pipeline (using 'Gr Liv Area' as the single feature)
# Create a specific preprocessor for just 'Gr Liv Area'
simple_lr_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Fit and transform only the 'Gr Liv Area' column
X_train_simple_lr = simple_lr_preprocessor.fit_transform(X_train[['Gr Liv Area']])
X_test_simple_lr = simple_lr_preprocessor.transform(X_test[['Gr Liv Area']])

simple_linear_model = LinearRegression()
print("\nTraining Simple Linear Regression (using Gr Liv Area)...")
simple_linear_model.fit(X_train_simple_lr, y_train)
y_pred_simple_linear = simple_linear_model.predict(X_test_simple_lr)
mse_simple_linear = mean_squared_error(y_test, y_pred_simple_linear)
r2_simple_linear = r2_score(y_test, y_pred_simple_linear)
print(f"Simple Linear Regression MSE: {mse_simple_linear:.4f}")
print(f"Simple Linear Regression R-squared: {r2_simple_linear:.4f}")

print("\nSample Simple Linear Regression Predictions vs. Actuals (first 5):")
comparison_df_simple_linear = pd.DataFrame({'Actual SalePrice': y_test.head(),
                                            'Predicted SalePrice (Simple LR)': y_pred_simple_linear[:5].round(2)})
print(comparison_df_simple_linear)


# 2. Multiple Linear Regression Pipeline (using all preprocessed features)
multiple_linear_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                           ('regressor', LinearRegression())])
print("\nTraining Multiple Linear Regression (using all features)...")
multiple_linear_pipeline.fit(X_train, y_train)
y_pred_multiple_linear = multiple_linear_pipeline.predict(X_test)
mse_multiple_linear = mean_squared_error(y_test, y_pred_multiple_linear)
r2_multiple_linear = r2_score(y_test, y_pred_multiple_linear)
print(f"Multiple Linear Regression MSE: {mse_multiple_linear:.4f}")
print(f"Multiple Linear Regression R-squared: {r2_multiple_linear:.4f}")

print("\nSample Multiple Linear Regression Predictions vs. Actuals (first 5):")
comparison_df_multiple_linear = pd.DataFrame({'Actual SalePrice': y_test.head(),
                                              'Predicted SalePrice (Multiple LR)': y_pred_multiple_linear[:5].round(2)})
print(comparison_df_multiple_linear)


# 3. Ridge Regression Pipeline with GridSearchCV for hyperparameter tuning
ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', Ridge())])
param_grid_ridge = {'regressor__alpha': np.logspace(-4, 4, 9)} # From 0.0001 to 10000, 9 values
grid_search_ridge = GridSearchCV(ridge_pipeline, param_grid_ridge, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
print("\nTraining Ridge Regression with GridSearchCV...")
grid_search_ridge.fit(X_train, y_train)
best_ridge_model = grid_search_ridge.best_estimator_
y_pred_ridge = best_ridge_model.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)
print(f"\nRidge Regression - Best Alpha: {grid_search_ridge.best_params_['regressor__alpha']:.6f}")
print(f"Ridge Regression - Best Cross-Validation MSE: {-grid_search_ridge.best_score_:.4f}")
print(f"Ridge Regression - Test Set MSE: {mse_ridge:.4f}")
print(f"Ridge Regression - Test Set R-squared: {r2_ridge:.4f}")

print("\nSample Ridge Regression Predictions vs. Actuals (first 5):")
comparison_df_ridge = pd.DataFrame({'Actual SalePrice': y_test.head(),
                                    'Predicted SalePrice (Ridge)': y_pred_ridge[:5].round(2)})
print(comparison_df_ridge)

print("\n--- Model Training Complete ---")

### Modeling (Simple, Multiple, and Ridge Regression)
- Data Preparation: Split the data into training and testing sets (X_train, X_test, y_train, y_test).
- Preprocessor Pipeline: Established a ColumnTransformer to handle different types of features: numerical features were imputed with medians and scaled; ordinal features were encoded with specified orders; and nominal features were one-hot encoded.

#### Simple Linear Regression:
- Focus: Trained using only the Gr Liv Area feature, demonstrating basic linear modeling with a single predictor.
- Preprocessing: Applied a specific imputer and scaler only to Gr Liv Area.
- Evaluation: Measured performance using Mean Squared Error (MSE) and R-squared
#### Multiple Linear Regression:
- Focus: Utilized all preprocessed features to train a standard LinearRegression model, representing a more complex linear relationship.
- Preprocessing: Used the comprehensive ColumnTransformer.
- Evaluation: Assessed performance with MSE and R-squared.
#### Ridge Regression:
-Focus: Implemented Ridge (L2 regularized) regression to combat overfitting and handle multicollinearity.
- Hyperparameter Tuning: Employed GridSearchCV with 5-fold cross-validation to search for the optimal alpha (regularization strength) within a defined range.
- Evaluation: Reported the best alpha, its cross-validation MSE, and test set MSE/R-squared.
- Outcome: Provided quantitative metrics and sample predictions for all three models, allowing for comparison of their performance

In [ ]:
plt.figure(figsize=(20, 6)) # Adjusted figure size for 3 plots

# Plot for Simple Linear Regression
plt.subplot(1, 3, 1)
plt.scatter(y_test, y_pred_simple_linear, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title(f"Simple Linear Regression\nR-squared: {r2_simple_linear:.2f}")
plt.grid(True)


# Plot for Multiple Linear Regression
plt.subplot(1, 3, 2)
plt.scatter(y_test, y_pred_multiple_linear, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title(f"Multiple Linear Regression\nR-squared: {r2_multiple_linear:.2f}")
plt.grid(True)


# Plot for Ridge Regression
plt.subplot(1, 3, 3)
plt.scatter(y_test, y_pred_ridge, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title(f"Ridge Regression\nR-squared: {r2_ridge:.2f}")
plt.grid(True)


plt.tight_layout()
plt.suptitle("Actual vs. Predicted SalePrice for Regression Models", y=1.03, fontsize=16)
plt.show()

# --- Residual Plots ---
plt.figure(figsize=(20, 6)) # Adjusted figure size for 3 plots

# Residual Plot for Simple Linear Regression
plt.subplot(1, 3, 1)
residuals_simple_linear = y_test - y_pred_simple_linear
plt.scatter(y_pred_simple_linear, residuals_simple_linear, alpha=0.7)
plt.hlines(0, y_pred_simple_linear.min(), y_pred_simple_linear.max(), colors='r', linestyles='--')
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residuals")
plt.title("Simple Linear Regression Residuals")
plt.grid(True)


# Residual Plot for Multiple Linear Regression
plt.subplot(1, 3, 2)
residuals_multiple_linear = y_test - y_pred_multiple_linear
plt.scatter(y_pred_multiple_linear, residuals_multiple_linear, alpha=0.7)
plt.hlines(0, y_pred_multiple_linear.min(), y_pred_multiple_linear.max(), colors='r', linestyles='--')
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residuals")
plt.title("Multiple Linear Regression Residuals")
plt.grid(True)


# Residual Plot for Ridge Regression
plt.subplot(1, 3, 3)
residuals_ridge = y_test - y_pred_ridge
plt.scatter(y_pred_ridge, residuals_ridge, alpha=0.7)
plt.hlines(0, y_pred_ridge.min(), y_pred_ridge.max(), colors='r', linestyles='--')
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residuals")
plt.title("Ridge Regression Residuals")
plt.grid(True)


plt.tight_layout()
plt.suptitle("Residual Plots for Regression Models", y=1.03, fontsize=16)
plt.show()

### Visual Analysis of Actual vs. Predicted Plots
- The top row of plots displays the Actual vs. Predicted SalePrice for each model. The red dashed line represents perfect predictions where Actual SalePrice equals Predicted SalePrice.
##### Simple Linear Regression:
- The points are quite scattered around the red line, indicating a moderate fit.
- The R-squared of 0.55 confirms that only about 55% of the variance in SalePrice is explained by Gr Liv Area alone. This suggests that while Gr Liv Area is an important feature, it's not sufficient by itself to accurately predict house prices.
##### Multiple Linear Regression:
- The points are much tighter around the red line, especially for lower and medium prices.
- The R-squared of 0.95 shows a significant improvement, indicating that approximately 95% of the variance in SalePrice is explained by the multiple features used. This model is performing much better at capturing the relationships in the data.
##### Ridge Regression:
- Similar to Multiple Linear Regression, the points are very tightly clustered around the red line, suggesting a strong fit.
- The R-squared of 0.95 is identical to the Multiple Linear Regression in this visualization, which is expected as Ridge is a form of linear regression. Its benefit often lies in generalization rather than drastically different in-sample R-squared.

### Visual Analysis of Residual Plots
- The bottom row of plots displays the Residuals (Actual - Predicted) vs. Predicted SalePrice. Ideally, residuals should be randomly scattered around zero with no discernible pattern, indicating that the model's errors are random and that it has captured most of the underlying patterns.
##### Simple Linear Regression Residuals:
- The residuals show a clear pattern, resembling a slight curve or fanning out, especially as predicted prices increase. This indicates heteroscedasticity (non-constant variance of errors) and suggests that the single Gr Liv Area feature is not sufficient to capture the complex relationships, and the model systematically over- or under-predicts at certain price ranges.
##### Multiple Linear Regression Residuals:
- The residuals are much more randomly distributed around zero compared to the simple linear model. This suggests that the model is capturing most of the linear relationships well.
- There might still be a slight fanning out towards higher predicted values, indicating some remaining heteroscedasticity or that the model struggles slightly more with predicting higher-priced homes.
##### Ridge Regression Residuals:
- The residual plot for Ridge Regression is very similar to Multiple Linear Regression, showing a generally random scatter around zero. This is a good sign, indicating that the regularization has not significantly altered the fundamental linear relationship captured by the multiple features but rather stabilized the coefficients.
- Any slight patterns observed in the Multiple Linear Regression plot are also present here, which is expected as both are linear models using the same features

In [ ]:
# K-Fold Cross-Validation Setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Define models to evaluate for K-Fold Cross-Validation
models_to_evaluate = {
    # Simple Linear Regression requires its own specialized preprocessor for the single feature
    "Simple Linear Regression": Pipeline(steps=[
        ('preprocessor', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])),
        ('regressor', LinearRegression())
    ]),
    "Multiple Linear Regression": Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())]),
    "Ridge Regression": Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge(alpha=best_ridge_model.named_steps['regressor'].alpha))])
}

print("\n--- Starting K-Fold Cross-Validation ---")

for name, model_pipeline in models_to_evaluate.items():
    mse_scores = []
    r2_scores = []

    print(f"\nEvaluating {name} with K-Fold Cross-Validation...")
    for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Special handling for Simple Linear Regression: select only 'Gr Liv Area'
        if name == "Simple Linear Regression":
            # For K-fold, we need to pass a DataFrame with only the selected feature to the pipeline's fit/transform
            # The pipeline for simple LR will then handle its own imputer/scaler
            model_pipeline.fit(X_train_fold[['Gr Liv Area']], y_train_fold)
            y_pred_fold = model_pipeline.predict(X_val_fold[['Gr Liv Area']])
        else:
            model_pipeline.fit(X_train_fold, y_train_fold)
            y_pred_fold = model_pipeline.predict(X_val_fold)

        mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))
        r2_scores.append(r2_score(y_val_fold, y_pred_fold))

    print(f"{name} - Average MSE: {np.mean(mse_scores):.4f} (Std: {np.std(mse_scores):.4f})")
    print(f"{name} - Average R-squared: {np.mean(r2_scores):.4f} (Std: {np.std(r2_scores):.4f})")

print("\n--- K-Fold Cross-Validation Complete ---")

### Quantitative Analysis of K-Fold Cross-Validation
- The K-Fold Cross-Validation results provide a more robust and generalized view of the models' performance by averaging scores across multiple splits of the data.
##### Simple Linear Regression:
- Average MSE: 3,023,035,288.38 (Standard Deviation: 348,892,597.16)
- Average R-squared: 0.5243 (Standard Deviation: 0.0302)
- Analysis: As expected, this model has a very high MSE and a low R-squared. The high standard deviation for MSE also indicates less stable performance across different folds. This confirms that using only one feature (Gr Liv Area) is insufficient for accurate house price prediction, which aligns with the visual analysis.
##### Multiple Linear Regression:
- Average MSE: 448,499,466.16 (Standard Deviation: 59,234,016.93)
- Average R-squared: 0.9282 (Standard Deviation: 0.0152)
- Analysis: This model shows a drastic improvement over simple linear regression, with a significantly lower MSE and a very high R-squared (over 92%). The standard deviation of both metrics is relatively low, indicating stable performance. This demonstrates the power of using multiple features.
##### Ridge Regression:
- Average MSE: 425,499,057.24 (Standard Deviation: 42,962,705.02)
- Average R-squared: 0.9320 (Standard Deviation: 0.0123)
- Analysis: Ridge Regression performs slightly better than Multiple Linear Regression in terms of average MSE and R-squared. More importantly, its standard deviation for both MSE and R-squared is lower than that of Multiple Linear Regression. This suggests that Ridge Regression, by applying L2 regularization, is slightly more robust and stable in its predictions across different data subsets. It effectively reduces the variance of the model, leading to more consistent performance on unseen data.